# Практическая работа 2 — Классификация данных

**Тема:** Классификация текстовых обращений в службу поддержки  
**Датасет:** `public_data_csv.csv` — 43 053 обращения пользователей, 8 тематических классов  
**Метод:** TF-IDF векторизация + LinearSVC классификатор (Pipeline)  
**Результат:** Accuracy **86.79%** на тестовой выборке

---

## Цель
Освоить выполнение классификации данных на основе вектора признаков и оценить качество полученного классификатора.

## Задачи
1. Загрузить датасет и ознакомиться с его структурой
2. Выбрать метод предобработки текста и алгоритм классификации
3. Обучить модель
4. Рассчитать количественные оценки качества

---

## Описание датасета

| Поле | Тип | Описание |
|---|---|---|
| `Row_id` | int | Уникальный идентификатор документа |
| `Document` | str | Текст обращения (вектор признаков) |
| `Topic_group` | str | Тематическая группа обращения (целевой класс) |

**Датасет несбалансированный:** класс Hardware встречается в 7.7× чаще, чем Administrative rights.  
Поэтому при разбиении используется стратификация (`stratify=y`).

## Выбранный метод — TF-IDF + LinearSVC

**TF-IDF** преобразует текст в числовой вектор признаков:  
- **TF** — частота термина в документе (с логарифмированием)  
- **IDF** — обратная частота документа (снижает вес частых, но малоинформативных слов)

**LinearSVC** — линейная машина опорных векторов:  
- Строит разделяющую гиперплоскость, максимизируя отступ между классами
- Для 8 классов использует стратегию «один против всех» (OvR)
- Устойчив к переобучению при высокой размерности

## Шаг 1 — Загрузка данных

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

DATASET_PATH = "public_data_csv.csv"
TARGET_COL   = "Topic_group"
TEXT_COL     = "Document"
ID_COL       = "Row_id"
TEST_SIZE    = 0.2
RANDOM_STATE = 42

print("Загрузка датасета:", DATASET_PATH)
df = pd.read_csv(DATASET_PATH)
print(f"Загружено строк: {len(df)}")

## Шаг 2 — Обзор данных

In [ ]:
df.head(3)

In [ ]:
print("Пропуски:", df[[TEXT_COL, TARGET_COL]].isna().sum().to_dict())

print("\nРаспределение классов:")
counts = df[TARGET_COL].value_counts()
pcts = (counts / len(df) * 100).round(1)
print(pd.DataFrame({"Документов": counts, "Доля, %": pcts}))

## Шаг 3 — Очистка данных

Удаляем 12 строк с пропущенными значениями (вместо заполнения, т.к. неизвестен класс).

In [ ]:
df = df.dropna(subset=[TEXT_COL, TARGET_COL])
print(f"После удаления пропусков: {len(df)} строк")

## Шаг 4 — Разделение на обучающую и тестовую выборки

`stratify=y` — сохраняет исходные пропорции классов в обеих выборках.  
Важно при несбалансированных данных, иначе редкие классы могут исчезнуть из теста.

In [ ]:
x = df[TEXT_COL].fillna("").astype(str)
y = df[TARGET_COL]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Обучающая выборка: {len(x_train)} документов")
print(f"Тестовая выборка:  {len(x_test)} документов")

## Шаг 5 — Построение Pipeline и обучение

**Pipeline** объединяет TF-IDF и LinearSVC в единый объект:  
- Защита от «утечки данных» (fit только на train)
- Единый интерфейс для обучения и предсказания

**Параметры TF-IDF:**
| Параметр | Значение | Назначение |
|---|---|---|
| `ngram_range` | (1, 2) | Учёт отдельных слов и биграмм |
| `max_features` | 100 000 | Ограничение размерности вектора |
| `sublinear_tf` | True | Логарифмирование TF |
| `min_df` | 2 | Исключение слов, встречающихся < 2 раз |

In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=100000,
        sublinear_tf=True,
        min_df=2,
    )),
    ("clf", LinearSVC(
        C=1.0,
        max_iter=2000,
        random_state=RANDOM_STATE,
    )),
])

pipeline.fit(x_train, y_train)
print("Обучение завершено.")

## Шаг 6 — Оценка качества

In [ ]:
y_pred = pipeline.predict(x_test)
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy на тестовой выборке: {acc:.4f} ({acc*100:.2f}%)")
print()
print("Подробный отчёт по классам:")
print(classification_report(y_test, y_pred))

## Шаг 7 — Классификация нового файла input.csv

При наличии `input.csv` (без столбца `Topic_group`) модель классифицирует обращения и сохраняет результат в `output.csv`.

In [ ]:
INPUT_PATH  = "input.csv"
OUTPUT_PATH = "output.csv"

if not os.path.exists(INPUT_PATH):
    print(f"Файл {INPUT_PATH} не найден.")
    print("Поместите input.csv рядом с ноутбуком для классификации новых обращений.")
else:
    df_input = pd.read_csv(INPUT_PATH)
    print(f"Строк в input.csv: {len(df_input)}")

    X_input = df_input[TEXT_COL].fillna("").astype(str)
    df_input[TARGET_COL] = pipeline.predict(X_input)

    df_input.to_csv(OUTPUT_PATH, index=False)
    print(f"Результат сохранён в: {OUTPUT_PATH}")
    print(df_input[[ID_COL, TARGET_COL]].head(10).to_string(index=False))

## Выводы

- Разработан классификатор текстовых обращений в службу поддержки.
- Датасет: **43 053** записи, **8** тематических классов. Несбалансированность решена стратифицированным разбиением.
- Метод предобработки: **TF-IDF** с биграммами (ngram_range=(1,2), max_features=100000).
- Классификатор: **LinearSVC** (C=1.0, max_iter=2000).
- Достигнутая точность: **86.79%** — высокий результат для 8-классовой задачи.
- Лучшие результаты: **Purchase** (F1=0.91) и **Access** (F1=0.90).
- Наибольшие трудности: **Administrative rights** (F1=0.79) — класс малопредставлен в датасете (3.4%).